In [0]:
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Reading notebook parameters")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  load_type         : '{load_type}'")
print(f"  merge_keys        : '{merge_keys}'")

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
source_filter = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading data from bronze")
print("=" * 80)
print(f"  Source table : {source_table}")
print(f"  Filter       : {source_filter}")
print(f"  Equivalent query:\n    SELECT * FROM {source_table} WHERE {source_filter}")

bronze_df = spark.read.table(source_table).filter(source_filter)

bronze_count = bronze_df.count()
print(f"  Records read from bronze : {bronze_count}")
print(f"  Source columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
if bronze_count == 0:
    print("  WARNING: No records found for the given landing_timestamp")
# display(bronze_df)

In [0]:
from pyspark.sql.functions import lit, current_timestamp
print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print("  Added columns : insert_timestamp = current_timestamp(), update_timestamp = current_timestamp()")
print(f"  Final columns ({len(updated_df.columns)}) : {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
load_type_norm = load_type.strip().upper()

print("=" * 80)
print("STEP 4: Writing data to silver")
print("=" * 80)
print(f"  Target table      : {target_table}")
print(f"  Load type         : {load_type_norm}")
print(f"  Target exists     : {spark.catalog.tableExists(target_table)}")
if spark.catalog.tableExists(target_table):
    print(f"  Target count (before) : {spark.read.table(target_table).count()}")
cols_sql = ", ".join([f"`{c}`" for c in updated_df.columns])

if load_type_norm == "FULL":
    # Full load: overwrite the target table
    print("  Mode: FULL -> overwriting the target table")
    print(f"  Equivalent query:\n    INSERT OVERWRITE {target_table}\n    SELECT {cols_sql} FROM source")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  Overwrite completed")

elif load_type_norm == "APPEND":
    # Append: add new records to the target table
    print("  Mode: APPEND -> appending records to the target table")
    print(f"  Equivalent query:\n    INSERT INTO {target_table} ({cols_sql})\n    SELECT {cols_sql} FROM source")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  Append completed")

elif load_type_norm == "MERGE":
    # Merge: update existing records and insert new ones based on merge keys
    print("  Mode: MERGE -> upserting records based on merge keys")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Parsed merge keys : {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = MERGE")

    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        print(f"  WARNING: merge keys not found in source columns: {missing_keys}")

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table
        print("  Target table does not exist -> creating it with the source data (initial load)")
        print(f"  Equivalent query:\n    CREATE TABLE {target_table} AS SELECT {cols_sql} FROM source")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Initial load completed")
    else:
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp
        update_set = {f"`{c}`": f"s.`{c}`" for c in updated_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in updated_df.columns}

        print(f"  Merge condition   : {merge_condition}")
        print(f"  Columns updated on match ({len(update_set)}) : {list(update_set.keys())}")
        print(f"  Columns inserted on no match ({len(insert_set)}) : {list(insert_set.keys())}")

        update_sql = ",\n      ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols_sql = ", ".join(insert_set.keys())
        insert_vals_sql = ", ".join(insert_set.values())
        merge_sql = (
            f"MERGE INTO {target_table} AS t\n"
            f"USING source AS s\n"
            f"ON {merge_condition}\n"
            f"WHEN MATCHED THEN UPDATE SET\n      {update_sql}\n"
            f"WHEN NOT MATCHED THEN INSERT ({insert_cols_sql})\n"
            f"VALUES ({insert_vals_sql})"
        )
        print("  Equivalent MERGE query:")
        print("    " + merge_sql.replace("\n", "\n    "))

        print("  Executing MERGE...")
        target_delta = DeltaTable.forName(spark, target_table)
        (
            target_delta.alias("t")
            .merge(updated_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute()
        )
        print("  MERGE completed")

        # Show merge metrics from the Delta transaction log
        metrics = target_delta.history(1).select("operationMetrics").collect()[0][0] or {}
        print(f"  Rows in source            : {metrics.get('numSourceRows')}")
        print(f"  Rows updated              : {metrics.get('numTargetRowsUpdated')}")
        print(f"  Rows inserted             : {metrics.get('numTargetRowsInserted')}")
        print(f"  Rows deleted              : {metrics.get('numTargetRowsDeleted')}")

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

print(f"  Target count (after) : {spark.read.table(target_table).count()}")
print("=" * 80)
print(f"Load to {target_table} finished successfully")
print("=" * 80)

In [0]:
record_count = updated_df.count()
print(f"STEP 5: Exiting notebook with processed record count = {record_count}")

dbutils.notebook.exit(str(record_count))

